# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import sys
sys.path.append('/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit')

check everything is working

# Constants

In [4]:
TAWJEEH_DATASET_NAME = 'ArabicMMLU'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/ArabicMMLU_experimental'
MODEL_PATH = "/raid_storage/shared_models/Qwen3-8B-Base"
MODEL_NAME = "Qwen3-8B"
TASK_NAME='NLU'

In [5]:
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [6]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14901,
  'tags': [],
  'name': 'A Simple Test Prompt',
  'task': {'name': 'dialect identification'},
  'status': 'DRAFT',
  'template': 'Please predict the most suitable dialect for the following text: {{arabic}}\xa0\r\n|||{{answer_choices[label]}}',
  'created_by': 'irfan',
  'dataset_name': 'arbml/AraBench_dev',
  'dataset_subset': 'default',
  'answer_choices': ['Tunisian',
   'MSA',
   'Morrocan',
   'Qatari',
   'Egyptian',
   'Lebanese'],
  'text_direction': 'ltr'},
 {'id': 14898,
  'tags': ['', 'Zero-shot COT'],
  'name': 'Prompt with zero-shot chain of thoughts',
  'task': {'name': 'claim verification'},
  'status': 'APPROVED',
  'template': "For the following task you have to label if the two sentences are of on of the following labels: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %} or {% endif %}{% endfor %}. Sentence 1: {{s1}}\xa0 and sentence 2: {{s2}}.\r\nLet's think step by step:\r\n|||\r\n{{answer_choices[label]}}",
  'created_by': 'ahmed',


In [7]:
len(prompts)

365

In [8]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

352

## Finetuning

### Get the dataset prompts

In [9]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

6

In [10]:
SELECTED_PROMPTS_IDS = [
    14571,
    14869,   
    14787,
    14797,
    14798,
    # 14799,
]

In [11]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the dataset

In [12]:
import datasets

In [13]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['ID', 'Source', 'Country', 'Group', 'Subject', 'Level', 'Question', 'Context', 'answer', 'A', 'B', 'C', 'D', 'E', 'is_few_shot'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['ID', 'Source', 'Country', 'Group', 'Subject', 'Level', 'Question', 'Context', 'answer', 'A', 'B', 'C', 'D', 'E', 'is_few_shot'],
        num_rows: 4575
    })
})

### Merge the prompts

In [14]:
from jinja2 import Environment, StrictUndefined

In [15]:
import re
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    return f'{prefix.strip()}\n{suffix.strip()}' # output is always the last line!

In [16]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        template = preprocess_template(template)
        sample['answer_choices'] = []
        for choice in prompt_template['answer_choices']:
            if choice in sample and sample[choice]:
                if isinstance(sample[choice], str) and sample[choice].strip():
                    sample['answer_choices'].append(choice)
                else:
                    sample[choice] = None
            else:
                sample[choice] = None
        # print(template)
        # print(prompt_template['id'])
        env = Environment(undefined=StrictUndefined)
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e

see how the template is applied on different examples

### Perform prompt-merge on one example prompt, for experimentation

In [17]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][3]))

This is a question. Select the correct answer!

Question: 
تكثر السيارات في

Choices:



A. المدينة



B. القرية



C. البادية




Answer:
A


In [18]:
step_size = len(hf_exp_dataset['train'])/len(dataset_prompts)
step_size

2000.0

In [19]:
rendered_train_prompts_dataset = list()
for i,sample in enumerate(tqdm(hf_exp_dataset['train'])):
    if i % step_size == 0:
        print(f'rending {dataset_prompts[int(i/step_size)]["template"]}','sample index:',i)
    rendered_train_prompts_dataset.append(
        apply_template(dataset_prompts[int(i/step_size)], sample)
    )
len(rendered_train_prompts_dataset)

  0%|          | 0/10000 [00:00<?, ?it/s]

rending This is a question. Select the correct answer!

Question: 
{{Question}}

Choices:
{% set choices = [A,B,C,D] %}
{% for choice in choices %}
{% if choice and choice.strip %}
{{ answer_choices[loop.index0] }}. {{choice}}
{% endif %}
{% endfor %}
Answer:
|||
{{answer_choices[answer_choices.index(answer)]}} sample index: 0


rending I am taking an MCQ test. Here is the question
{{Question}}
What is the answer given the following choices:
{% set answers = [A, B, C, D, E] %}
{% for answer in answers %} {% if answer != None %}  {{ answer_choices[loop.index0] }}. {{ answer }} {% endif %}{% endfor %}
|||
{{answer}} sample index: 2000


rending You are given the following choices:
{% set answers = [A, B, C, D, E] %}
{% for answer in answers %} {% if answer != None %}  {{ answer_choices[loop.index0] }}. {{ answer }} {% endif %}
{% endfor %}
Answer the following question: {{Question}}
|||
{{answer}} sample index: 4000


rending You are tasked with solving a multiple-choice question (MCQ). Your goal is to analyze the question, consider all options carefully, and provide the correct answer.
Here is the MCQ question: {{Question}}
And here are the answer options:
{% if A %}
{{answer_choices[0]}}. {{A}} 
{% endif %}
{% if B %}
{{answer_choices[1]}}. {{B}} 
{% endif %}
{% if C %}
{{answer_choices[2]}}. {{C}} 
{% endif %}
{% if D %}
{{answer_choices[3]}}. {{D}} 
{% endif %}
{% if E %}
{{answer_choices[4]}}. {{E}}
{% endif %}
The correct answer among the above options is: 
|||
{{answer_choices[["A", "B", "C", "D", "E"].index(answer)]}} sample index: 6000


rending You have the following question: {{Question}} and the following options: {% if A %} {{answer_choices[0]}}. {{A}} {% endif %} {% if B %} {{answer_choices[1]}}. {{B}} {% endif %} {% if C %} {{answer_choices[2]}}. {{C}} {% endif %} {% if D %} {{answer_choices[3]}}. {{D}} {% endif %} {% if E %} {{answer_choices[4]}}. {{E}} {% endif %}, the correct answer is 
|||
{{answer_choices[["A", "B", "C", "D", "E"].index(answer)]}} sample index: 8000


10000

## Finetune the LLM

In [20]:
GLOBAL_SEED = 42

In [21]:
import random
random.seed(GLOBAL_SEED)

In [22]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, Qwen3Initializer, LoRAConfigRepository
from sklearn.model_selection import train_test_split

🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py


/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit/llm/message_generator.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [23]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=Qwen3Initializer(),
)
llm_loader

In [24]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /raid_storage/shared_models/Qwen3-8B-Base/config.json


`torch_dtype` is deprecated! Use `dtype` instead!


Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 12288,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_at

loading weights file /raid_storage/shared_models/Qwen3-8B-Base/model.safetensors.index.json


Instantiating Qwen3ForCausalLM model under default dtype torch.bfloat16.


...

loading file tokenizer_config.json


loading file chat_template.jinja


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


loading configuration file /raid_storage/shared_models/Qwen3-8B-Base/generation_config.json


Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "max_new_tokens": 2048
}



In [25]:
import re

train_samples,eval_samples = train_test_split(
    rendered_train_prompts_dataset,
    test_size=0.1,
    random_state=GLOBAL_SEED,
)

def generate_tuple(sample):
    sample_lines = sample.splitlines()
    # each sample should be two lines only, the first line is for the inputs and the last is the output
    # we did the join for generalization
    prefix = '\n'.join(sample_lines[:-1])
    prefix = prefix.strip()
    # prefix = re.sub(r'\s+', ' ', prefix).strip()
    # outputs are always the last line
    suffix = sample_lines[-1].strip()
    suffix = f' {suffix}' # adding this space is important to split between input and output
    return prefix,suffix

train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

(9000,
 1000,
 [('You are given the following choices:\n\n   A. الجمھور \n   B. - الأحزاب \n   C. النقابات \n \n \n\nAnswer the following question: من أھم الفواعل غیر رسمیة للسیاسة العامة: وسائل الاعلام و.....',
   ' A'),
  ('You are given the following choices:\n\n   A. لعن رسول الله شارب الخمر وحاملها وعاصرها وبائعها \n   B. لعن آكل الربا \n   C. لعن المتشبهين من الرجال بالنساء، ولعن المتشبهات من النساء بالرجال \n   D. جميع ما ذكر \n \n\nAnswer the following question: أذكر مثالاً على بعض الأفعال التي خصها النبي باللعن على عمومها دون غيرها، ودون أن يسمى أصحابها',
   ' D'),
  ('This is a question. Select the correct answer!\n\nQuestion: \nتعتبر براءات الاختراع و انظمة المعلومات و العلامات التجارية من الاصول :\n\nChoices:\n\n\n\nA. الحقيقية\n\n\n\nB. المالية\n\n\n\nC. الملموسة\n\n\n\nD. غير الملموسة\n\n\nAnswer:',
   ' D'),
  ('This is a question. Select the correct answer!\n\nQuestion: \n……. تلبية حاجات السكان من الإنتاج الزراعي محليا دون اللجوء إلى استيرادها من الخارج. \n\nChoices:\n\

In [26]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.llama_3(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=16,
    eval_batch_size=16,
    output_dir=f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}'
)

PyTorch: setting up devices


The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit/llm/llm_trainer.py:83: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.


Using auto half precision backend



***** Running Evaluation *****


...

  Num examples = 1000


  Batch size = 16


{'eval_loss': 0.8958123326301575, 'eval_model_preparation_time': 0.0002, 'eval_runtime': 14.8153, 'eval_samples_per_second': 67.498, 'eval_steps_per_second': 4.252, 'epoch': 3.108348134991119}




Training completed. Do not forget to share your model on huggingface.co/models =)




0.42911139130592346

In [27]:
exit()